In [ ]:
#香川県内に本社・本店がある会社(協同組合や社団法人は含めない)を抽出する(関数にまとめた)
import pandas as pd
import openpyxl

#①国交省からDLした建設関連業の登録業者(20260411にDL実施)のExcelファイルを読み込み
df = pd.read_excel('建設関連業の登録業者(測量業者_260331).xlsx')

def construction_contractor(df):
    #②前準備
    #営業所区分コードに欠測値を補い、かつ文字型に変換
    df['営業所区分コード'] = pd.to_numeric(df['営業所区分コード'], errors='coerce').astype('int64')

    #③香川県の会社を検索・抽出
    #営業所区分コードが1(本店や本社)でかつ会社種別コードが2,3,4,5のものを抽出
    sales_office =df[(df['営業所区分コード']==1)&(df['会社種別コード'].isin([2,3,4,5]))]

    #本社や本店が香川県にある会社を検索
    kagawa_company = sales_office[(sales_office['営業所所在地'].str.contains('香川県',na=False))]
    return kagawa_company

result = construction_contractor(df)
#④新規Excelファイル作成・書込
writer = pd.ExcelWriter('建設関連業の登録業者(測量業者)_260331_香川の会社のみ_p_関数.xlsx')
result.to_excel(writer,sheet_name='香川県内の会社一覧', index=False)
writer.close()

In [4]:
#香川県内の測量会社の完成業務収入(建設会社における売上高に該当する項目)を集計
import pandas as pd
import openpyxl

#①前準備
#整理したExcelファイルを読み込んで、データフレーム化
df = pd.read_excel('建設関連業の登録業者(測量業者)_260331_香川の会社のみ_p.xlsx')

#必要な列のみをコピー
new_sheet1 = df.loc[:, ['商号又は名称', '商号又は名称かな',
    '法人＿完成業務収入', '役員兼＿技術＿計','その他＿技術＿計']]

#列名を変更
new_sheet1 = new_sheet1.rename(columns={'法人＿完成業務収入':'完成業務収入(円)',
    '役員兼＿技術＿計':'技術社員(役員)(人)','その他＿技術＿計':'技術社員(役員以外)(人)'})

#技術職員(人)=技術職員(役員)＋技術職員(役員以外)
new_sheet1['全技術社員(人)'] = new_sheet1['技術社員(役員)(人)']+new_sheet1['技術社員(役員以外)(人)']

#空のシートを作成
#シート2は、香川県の会社全体の完成業務収入に対する各会社のシェア率を算出
new_sheet2 = pd.DataFrame()
new_sheet2 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)']]
#シート3は、技術社員1人あたりの各会社の完成業務収入を算出
new_sheet3 = pd.DataFrame()
new_sheet3 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)','全技術社員(人)']]

#②集計
#完成業務収入(completed_contract_revenue)の合計
completed_contract_revenue = new_sheet2['完成業務収入(円)'].sum()

#③シェア算出・順位付け
#会社ごとの完成業務収入のシェア
new_sheet2['シェア率(%)'] = (new_sheet2['完成業務収入(円)']/completed_contract_revenue) * 100
#完成業務収入のシェア率による順位付け
new_sheet2['順位'] = new_sheet2['シェア率(%)'].rank(ascending=False)
new_sheet2['順位'] = new_sheet2['順位'].astype(int)

#シート2の一番下の行に、完成業務収入の合計値を書き込む
#空白行を1行分作成する
empty_row2 = pd.DataFrame([{}])
new_sheet2 = pd.concat([new_sheet2, empty_row2], ignore_index=False)
#合計値を書き込む
new_sheet2.at[len(new_sheet2)+1, '商号又は名称かな'] = '完成業務収入合計(円)'
new_sheet2.at[len(new_sheet2), '完成業務収入(円)'] = completed_contract_revenue

#技術社員1人ごとの完成業務収入ランキング
#技術社員1人ごとの完成業務収入算出
new_sheet3['1人あたりの完成業務収入(円)'] = (new_sheet3['完成業務収入(円)']/new_sheet3['全技術社員(人)'])
#技術社員1人あたりの完成業務収入による順位付け
new_sheet3['順位'] = new_sheet3['1人あたりの完成業務収入(円)'] .rank(ascending=False)
new_sheet3['順位'] = new_sheet3['順位'].astype(int)

#シート3の一番下の行に、完成業務収入の合計値を書き込む
#空白行を1行分作成する
empty_row3 = pd.DataFrame([{}])
#合計値を書き込む
new_sheet3 = pd.concat([new_sheet3, empty_row3], ignore_index=False)
new_sheet3.at[len(new_sheet3)+1, '商号又は名称かな'] = '完成業務収入合計(円)'
new_sheet3.at[len(new_sheet3), '完成業務収入(円)'] = completed_contract_revenue

#④新規Excelファイル作成・書込
with pd.ExcelWriter('建設関連業の登録業者(測量業者)_260331_香川の会社のみ_シェア_p2.xlsx') as writer:
    new_sheet1.to_excel(writer,sheet_name='香川県内の会社一覧_整理', index=False)
    new_sheet2.to_excel(writer,sheet_name='完成業務収入のシェア率', index=False)
    new_sheet3.to_excel(writer,sheet_name='技術社員1人当たりの完成業務収入', index=False)

In [ ]:
#香川県内の測量会社の完成業務収入(建設会社における売上高に該当する項目)を集計(集計結果の挿入方法でエラー)
import pandas as pd
import openpyxl

#①前準備 #整理したExcelファイルを読み込んで、データフレーム化
df = pd.read_excel('建設関連業の登録業者(測量業者)_260331_香川の会社のみ_p.xlsx')

#必要な列のみをコピー
new_sheet1 = df.loc[:, ['商号又は名称', '商号又は名称かな', '法人＿完成業務収入',
    '役員兼＿技術＿計','その他＿技術＿計']]
#列名を変更
new_sheet1 = new_sheet1.rename(columns={'法人＿完成業務収入':'完成業務収入(円)',
    '役員兼＿技術＿計':'技術社員(役員)(人)', 'その他＿技術＿計':'技術社員(役員以外)(人)'})
#技術職員(人)=技術職員(役員)＋技術職員(役員以外)
new_sheet1['全技術社員(人)'] = new_sheet1['技術社員(役員)(人)']+new_sheet1['技術社員(役員以外)(人)']

#空のシートを作成
#シート2は、香川県の会社全体の完成業務収入に対する各会社のシェア率を算出
new_sheet2 = pd.DataFrame()
new_sheet2 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)']]
#シート3は、技術社員1人あたりの各会社の完成業務収入を算出
new_sheet3 = pd.DataFrame()
new_sheet3 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)',
    '全技術社員(人)']]

#②集計
#完成業務収入(completed_contract_revenue)の合計
completed_contract_revenue = new_sheet1['完成業務収入(円)'].sum()

#③シェア算出・順位付け #会社ごとの完成業務収入のシェア
new_sheet2['シェア率(%)'] = (new_sheet1['完成業務収入(円)']/completed_contract_revenue) * 100

#完成業務収入のシェア率による順位付け
new_sheet2['順位'] = new_sheet2['シェア率(%)'].rank(ascending=False)
new_sheet2['順位'] = new_sheet2['順位'].astype(int)

#シート2の一番下の行に、完成業務収入の合計値を書き込む
#空白行を1行分作成する
new_index =new_sheet2.shape[0] + 2
#合計値を書き込む
new_sheet2.at[new_index,'商号又は名称かな']='完成業務収入合計(円)'
new_sheet2.at[new_index,'完成業務収入(円)']=completed_contract_revenue

#技術社員1人ごとの完成業務収入ランキング
#技術社員1人ごとの完成業務収入算出
new_sheet3['1人あたりの完成業務収入(円)'] = (new_sheet3['完成業務収入(円)']/new_sheet3['全技術社員(人)'])
#技術社員1人あたりの完成業務収入による順位付け
new_sheet3['順位'] = new_sheet3['1人あたりの完成業務収入(円)'] .rank(ascending=False)
new_sheet3['順位'] = new_sheet3['順位'].astype(int)

#シート3の一番下の行に、完成業務収入の合計値を書き込む
#空白行を1行分作成する
new_index = new_sheet3.shape[0]+2
#合計値を書き込む
new_sheet3.at[new_index, '商号又は名称かな'] = '完成業務収入合計(円)'
new_sheet3.at[new_index, '完成業務収入(円)'] = completed_contract_revenue

#④新規Excelファイル作成・書込
with pd.ExcelWriter('建設関連業の登録業者(測量業者)_260331_香川の会社のみ_シェア_p_miss.xlsx') as writer:
    new_sheet1.to_excel(writer,sheet_name='香川県内の会社一覧_整理', index=False)
    new_sheet2.to_excel(writer,sheet_name='完成業務収入のシェア率', index=False)
    new_sheet3.to_excel(writer,sheet_name='技術社員1人当たりの完成業務収入', index=False)